# Gradient boosting — perinatal_anc cohort

Train **gradient boosting** on the `perinatal_anc` cohort and tune it with cluster-grouped
cross-validation on the training rows. The test split is scored on Sunday only.

Owner: Jeff  ·  **`random_state=42` so results reproduce.**

## 1 — Load the data

In [ ]:
import sys
from pathlib import Path

# make the repo root importable when run from ml/notebooks/modeling
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml" / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ml.src.dataset import build_cohort, cv_folds, SELECTED, OUTCOMES

COHORT = "perinatal_anc"
# load the combined, cleaned cohort
df = build_cohort(COHORT)

# keep training rows only; the test split stays closed until Sunday
train = df[df["split"] == "train"].reset_index(drop=True)
print(f"train rows: {len(train):,}  clusters: {train['cluster_id'].nunique():,}")

## 2 — Features and target
Features are the cohort's `SELECTED` columns minus the outcome and the design columns.

In [ ]:
TARGET = OUTCOMES[COHORT]
DESIGN = ["split", "cluster_id", "country", "weight"]

# features = SELECTED minus outcome and design columns
FEATURES = [c for c in SELECTED[COHORT] if c not in DESIGN + [TARGET]]
X_train, y_train = train[FEATURES], train[TARGET]

# event count and rate (the constant-baseline PR-AUC)
print(f"{len(FEATURES)} features: {FEATURES}")
print(f"events: {int(y_train.sum()):,}  rate: {y_train.mean():.2%}")
# blanks per feature (first pregnancies have no interval by design)
print(X_train.isna().sum()[lambda s: s > 0].to_dict())

## 3 — Train and tune
`HistGradientBoostingClassifier`: supports `class_weight='balanced'` for the rare outcome
(no resampling) and handles blanks natively, so first-pregnancy intervals stay blank.
Tuned with `GridSearchCV` on `cv_folds()` (5 cluster-grouped folds), PR-AUC first.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# balanced class weights for the imbalance; no early stopping so max_iter is tuned
gb = HistGradientBoostingClassifier(class_weight="balanced", early_stopping=False, random_state=42)
# all features are numeric, so the pipeline is just the model
pipe = Pipeline([("model", gb)])

# small grid of simple models: 48 settings x 5 folds
GRID = {
    "model__learning_rate": [0.02, 0.05],
    "model__max_iter": [50, 100],
    "model__max_leaf_nodes": [4, 7],
    "model__min_samples_leaf": [20, 80, 150],
    "model__l2_regularization": [1.0, 5.0],
}

# cluster-grouped folds from dataset.py
folds = cv_folds(train, COHORT)
search = GridSearchCV(pipe, GRID, cv=folds, scoring=["average_precision", "roc_auc"],
                      refit="average_precision", n_jobs=1)
# fit on training rows only
search.fit(X_train, y_train)

In [ ]:
# per-fold scores of the best setting
res, k = search.cv_results_, search.best_index_
fold_scores = pd.DataFrame({
    "pr_auc": [res[f"split{i}_test_average_precision"][k] for i in range(len(folds))],
    "roc_auc": [res[f"split{i}_test_roc_auc"][k] for i in range(len(folds))],
    # constant baseline: PR-AUC of a flat score = event rate in the fold
    "baseline_pr_auc": [y_train.iloc[val].mean() for _, val in folds],
}, index=pd.Index(range(1, len(folds) + 1), name="fold"))

# mean and sd over folds
summary = fold_scores.agg(["mean", "std"]).T.round(4)
print(summary)
print("best params:", {p.replace("model__", ""): v for p, v in search.best_params_.items()})

### CV results (training rows only)

- PR-AUC **0.079 ± 0.027** over 5 cluster-grouped folds, against a constant baseline of 0.031 (about 2.5×).
- ROC-AUC **0.625 ± 0.039**.
- Best setting: learning_rate 0.05, max_iter 50, max_leaf_nodes 4, min_samples_leaf 80, l2_regularization 1.
- A wider grid (up to 300 trees, 15 leaves) scored lower (PR-AUC 0.072), so simpler models suit this cohort.
- Fold scores vary most here (sd 0.027): only 412 events in the training rows.

In [ ]:
# fold scores against the constant baseline
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(fold_scores.index, fold_scores["pr_auc"], label="gradient boosting")
axes[0].plot(fold_scores.index, fold_scores["baseline_pr_auc"], "k--o", label="constant baseline")
axes[0].set(title="CV PR-AUC by fold", xlabel="fold", ylabel="PR-AUC")
axes[0].legend()
axes[1].bar(fold_scores.index, fold_scores["roc_auc"])
axes[1].axhline(0.5, color="k", ls="--")
axes[1].set(title="CV ROC-AUC by fold", xlabel="fold", ylabel="ROC-AUC", ylim=(0.4, 1))
fig.suptitle(f"{COHORT}: gradient boosting, best grid setting")
fig.tight_layout()
plt.show()

## 4 — Evaluate on the test split

**Sunday: test split.** Refit the best setting on all training rows, then report PR-AUC (headline),
ROC-AUC and recall at a fixed flag rate, against the checklist baseline (`checklist_baseline.ipynb`).

## 5 — Save the result

**Sunday: test split.** Write one row to `model_metrics` with `db.metrics.save_metrics(...)`,
and save the fitted model with `joblib.dump` to `ml/models/` if it is the cohort winner.